# Fermionic 3D toric-code sign-head — in-vivo speed ladder

Speed only (accuracy/ceiling figures live in `fermionic_signhead_figures.ipynb`): in-vivo VMC
step time vs qubit count $N$ for the five sign-head arms (`baseline` = no head, `cup` = exact
cup-product h=0 sign, `linear`/`vote`/`pt2` = the decoded heads), measured during real training
(`tc3d/train.py`'s per-step `timing` phase breakdown, `tc3d/sign_frame.py`'s `t_head`/
`n_head_configs` host-side instrumentation, `tc3d/sign_decoders.py`'s per-row lit-line/fallback
counts). Data: `results/fermionic_speed/speed_L{L}_{ARM}_hx{HX}_hz{HZ}_s{SEED}.json` (+ its
periodic `.curve.json` twin), rsynced from the cluster; branch `feat/signhead-speed`. Mirrors the
2D peer's `06_decoder_scaling.ipynb` Figure 1 layout (plateau-mean head time / share / total step
time vs $N$, log–log, baseline drawn thick grey).

All inputs are read by glob, so this degrades gracefully (prints, no crash) while
`results/fermionic_speed/` is still empty or partially landed — re-execute after each
`cluster.sh fetch`. `savefig` lines stay commented out (the user saves figures by hand).

## 1. Setup / CONFIG

In [ ]:
import glob
import json
import re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator, NullLocator, ScalarFormatter

# house plot style (shared with fermionic_signhead_figures.ipynb / fermionic_figs.py)
plt.rcParams.update({
    "figure.dpi": 120, "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "legend.frameon": False,
})

ROOT = Path.cwd().resolve().parents[1]      # analysis/notebooks -> repo root
DATA_DIR = ROOT / "results" / "fermionic_speed"
FIGS = ROOT / "analysis" / "figs"

PLATEAU_START = 20        # 0-based index into the per-step `timing` list (JIT + transient excluded)
DEFAULT_N_SAMPLES = 4096  # reference batch size; a run at a different n_samples is flagged, not
                          # silently averaged in alongside the rest of its curve (Figure 1)

ARMS = ["baseline", "cup", "linear", "vote", "pt2"]
ARM_LABEL = {
    "baseline": "no head (GPU-only)",
    "cup": "cup product (on-support form only)",
    "linear": "fixed representative (GF(2)-linear)",
    "vote": "majority over minimal recoveries (1st-order PT)",
    "pt2": "PT with 2nd-order tie-break",
}
# Okabe-Ito; baseline drawn thick grey with an x-marker, like the 2D peer's "no head" arm
ARM_COLOR = {"baseline": "0.35", "cup": "#0072B2", "linear": "#E69F00", "vote": "#D55E00", "pt2": "#009E73"}
ARM_MARK = {"baseline": "x", "cup": "o", "linear": "s", "vote": "^", "pt2": "D"}

POINTS = [(0.5, 0.2), (0.2, 0.0)]    # (stress, contrast) field points

# N = 3 L^2 (L-1) qubits, OBC cubes
N_OF_L = {2: 12, 3: 54, 4: 144, 5: 300, 6: 540}
SIZES = sorted(N_OF_L)               # the full expected L-ladder

## 2. Loader

**File preference.** `speed_L*_s*.json` matches BOTH the final result json
(`speed_..._s{SEED}.json`, written once after `run_loop` returns) and its own periodic-checkpoint
twin (`speed_..._s{SEED}.curve.json`, whose own suffix is also `.json`). `tc3d/train.py` writes
the periodic checkpoint's `curve` dict *inside* `on_step`, which runs **before** `on_timing`
appends that same step's timing entry — so a `.curve.json` always lags the true timing list by
one entry, even for a run that finished perfectly cleanly (an $n_iter=60$ run's `.curve.json`
ends at timing step 58, not 59). The loader therefore globs once, sorts each stem's candidates
so the final json sorts first (`p.name.endswith(".curve.json")` as the tie-break key) and keeps
only the first match per stem: **the final json wins whenever it exists**, and a run falls back
to its `.curve.json` **only** when no final json was ever written for that stem — which is
itself the signature of a run killed by wall-clock before it could finish (flagged
`walltime_cut = True`, kept and averaged over whatever window it has, not dropped). A run with a
final json is therefore *never* `walltime_cut`; its `diverged` column instead comes straight from
the final json's own `diverged` field (`None`/unknown for a `walltime_cut` run, since that field
is never written to a bare `.curve.json`).

(The `.curve.json` lag itself is now fixed at the source -- `on_timing` runs before `on_step`'s
checkpoint write, so a freshly-produced `.curve.json` no longer lags. The final-json-first
preference stays regardless, both because it is a strict improvement for old already-landed files
that still carry the lag, and because "no final json at all" remains the correct `walltime_cut`
signature either way.)

**Window.** Within whichever file won, the plateau window is `timing[PLATEAU_START:]` — a
0-based slice by list index, not by training step number. A run is **dropped entirely** when that
window has fewer than 5 entries.

Per surviving run: `step_wall`/`t_head` = window **means** of `total`/`t_head` (`baseline` has no
`t_head` key at all — treated as 0, it carries no head); `step_wall_median`/`t_head_median` = the
window **medians** of the same two quantities (reported alongside the means because a mean is not
robust to a one-off stall); `outliers` = the count of window entries whose `total` exceeds 5x the
window's median `total` — a mid-run JIT recompile shows up as exactly this: one entry far above
the plateau, dragging the *mean* up while the *median* stays put. `share = t_head / step_wall`
(the published, plotted statistic); since the SR solve is an iterative CG (`--qgt onthefly`), the
`qgt` phase is itself data-dependent per arm/step, so `share`'s denominator carries a data-
dependent (typically a few percent) CG contribution on top of the head. `qgt_mean` reports that
phase directly, and `share_noqgt = mean(t_head) / mean(total - qgt)` is the head-share with that
CG term divided back out — not plotted (panel (b) stays on `share`, the published quantity), but
in the table for comparison. `t_head_per_row_us = 1e6 * sum(t_head) / sum(n_head_configs)` over
the window — the **size- and batch-invariant** per-row head cost, unaffected by `n_samples`, so a
reduced-batch point (e.g. `pt2` at `n_samples=512`) is still directly comparable to a
`n_samples=4096` point on this one column even though its raw `t_head`/`step_wall`/panel curves
are not; `k_mean = sum(k_sum)/sum(n_head_configs)` (mean lit lines per row); `fallback_frac`/
`pt2_frac`/`tie_frac` = `sum(n_fallback|n_pt2|n_tie) / sum(n_head_configs)`; `drift_pct` = the
least-squares slope of `t_head` over the window as a percent of its mean. `baseline` (and any arm
without `k_sum`/`n_head_configs`) gets `NaN` for the head-composition fields rather than a
spurious 0.

In [ ]:
FNAME_RE = re.compile(
    r"^speed_L(?P<L>\d+)_(?P<arm>" + "|".join(ARMS) + r")_hx(?P<hx>[0-9.]+)_hz(?P<hz>[0-9.]+)_s(?P<seed>\d+)$"
)


def _num_qubits(L):
    return N_OF_L.get(L, 3 * L ** 2 * (L - 1))


def _stem(path):
    name = path.name
    if name.endswith(".curve.json"):
        return name[: -len(".curve.json")]
    if name.endswith(".json"):
        return name[: -len(".json")]
    return name


def _frac(window, num_key):
    num = np.array([w.get(num_key, np.nan) for w in window], float)
    den = np.array([w.get("n_head_configs", np.nan) for w in window], float)
    if np.isnan(num).any() or np.isnan(den).any() or np.nansum(den) == 0:
        return float("nan")
    return float(np.sum(num) / np.sum(den))


def load_speed_runs(data_dir=DATA_DIR, plateau_start=PLATEAU_START):
    """Glob speed_*.json files under data_dir; return one tidy dict per
    surviving run (see the file-preference and window rules above). Tolerant
    of a missing/empty/partial directory -- never raises."""
    data_dir = Path(data_dir)
    if not data_dir.is_dir():
        print(f"[loader] {data_dir} does not exist yet -- 0 runs found.")
        return []

    # `speed_L*_s*.json` matches both a final json and its `.curve.json` twin;
    # sort each stem's candidates final-json-first, then keep the first per stem.
    all_files = sorted(data_dir.glob("speed_L*_s*.json"),
                        key=lambda p: (_stem(p), p.name.endswith(".curve.json")))
    paths, seen_stem = [], set()
    for f in all_files:
        stem = _stem(f)
        if stem in seen_stem:
            continue
        seen_stem.add(stem)
        paths.append(f)

    runs, dropped, unparsed = [], [], []
    for f in paths:
        m = FNAME_RE.match(_stem(f))
        if not m:
            unparsed.append(f.name)
            continue
        L, arm = int(m["L"]), m["arm"]
        hx, hz, seed = float(m["hx"]), float(m["hz"]), int(m["seed"])
        is_final = not f.name.endswith(".curve.json")

        d = json.load(open(f))
        cfg = d.get("config", {})
        timing = d.get("curve", {}).get("timing", [])
        n_samples = cfg.get("n_samples")
        # only reached the bare .curve.json when no final json exists for this
        # stem -- that absence IS the walltime-cut signature (see markdown above)
        walltime_cut = not is_final
        diverged = bool(d.get("diverged")) if is_final else None

        window = timing[plateau_start:]
        if len(window) < 5:
            dropped.append((f.name, len(window)))
            continue

        total = np.array([w.get("total", np.nan) for w in window], float)
        th = np.array([w.get("t_head", 0.0) for w in window], float)
        qgt = np.array([w.get("qgt", np.nan) for w in window], float)
        n_hc = np.array([w.get("n_head_configs", np.nan) for w in window], float)
        mean_total = float(np.nanmean(total))
        mean_th = float(np.nanmean(th))
        mean_qgt = float(np.nanmean(qgt))
        share = mean_th / mean_total if mean_total > 0 else float("nan")
        mean_total_noqgt = float(np.nanmean(total - qgt))
        share_noqgt = mean_th / mean_total_noqgt if mean_total_noqgt > 0 else float("nan")
        if np.isfinite(n_hc).all() and np.nansum(n_hc) > 0:
            t_head_per_row_us = 1e6 * float(np.sum(th)) / float(np.sum(n_hc))
        else:
            t_head_per_row_us = float("nan")

        # medians (robust to a one-off stall) + an outlier count off the mean
        step_wall_median = float(np.nanmedian(total))
        t_head_median = float(np.nanmedian(th))
        outliers = int(np.nansum(total > 5 * step_wall_median)) if step_wall_median > 0 else 0

        n_win = len(window)
        if mean_th > 0 and n_win >= 2:
            slope = float(np.polyfit(np.arange(n_win), th, 1)[0])
            drift_pct = 100.0 * slope * n_win / mean_th
        else:
            drift_pct = float("nan")

        runs.append(dict(
            name=_stem(f), size=L, N=_num_qubits(L), arm=arm, hx=hx, hz=hz, seed=seed,
            n_samples=n_samples, step_wall=mean_total, step_wall_median=step_wall_median,
            t_head=mean_th, t_head_median=t_head_median, t_head_per_row_us=t_head_per_row_us,
            share=share, share_noqgt=share_noqgt, qgt_mean=mean_qgt, drift_pct=drift_pct,
            k_mean=_frac(window, "k_sum"), fallback_frac=_frac(window, "n_fallback"),
            pt2_frac=_frac(window, "n_pt2"), tie_frac=_frac(window, "n_tie"),
            outliers=outliers, n_steps=n_win, walltime_cut=walltime_cut, diverged=diverged,
        ))

    if unparsed:
        print(f"[loader] {len(unparsed)} file(s) did not match the naming schema: {unparsed[:5]}")
    if dropped:
        print(f"[loader] dropped {len(dropped)} run(s) with <5 window entries: {dropped}")
    runs.sort(key=lambda r: (r["hx"], r["hz"], ARMS.index(r["arm"]) if r["arm"] in ARMS else 99, r["N"]))
    print(f"[loader] {len(runs)} run(s) loaded from {data_dir}")
    return runs

In [ ]:
# NOTE: pandas is not installed in this repo's shared .venv (numpy/scipy/numba/netket
# only -- see CLAUDE.md); rather than pip-installing into the shared venv unasked, the
# tidy table below is a plain list of per-run dicts (one row per run, same schema a
# pd.DataFrame(RUNS) would give if pandas is ever added).
RUNS = load_speed_runs()

_COLS = [("size", 4, "d"), ("N", 4, "d"), ("arm", 8, "s"), ("point", 12, "s"),
         ("n_samples", 6, "s"),
         ("step_wall", 10, ".4f"), ("step_wall_median", 10, ".4f"),
         ("t_head", 8, ".4f"), ("t_head_median", 8, ".4f"), ("t_head_per_row_us", 9, ".3f"),
         ("share", 7, ".3f"), ("share_noqgt", 8, ".3f"), ("qgt_mean", 8, ".4f"),
         ("drift_pct", 8, ".1f"), ("k_mean", 7, ".3f"), ("fallback_frac", 10, ".2e"),
         ("outliers", 8, "d"), ("n_steps", 6, "d"), ("walltime_cut", 4, "s"), ("diverged", 4, "s")]


def print_table(runs):
    header = " ".join(f"{name:>{w}}" for name, w, _ in _COLS)
    print(header)
    for r in runs:
        row = dict(r, point=f"({r['hx']:g},{r['hz']:g})",
                    n_samples=str(r["n_samples"]) if r["n_samples"] is not None else "?",
                    walltime_cut="Y" if r["walltime_cut"] else "",
                    diverged={True: "Y", False: "", None: "?"}[r["diverged"]])
        print(" ".join(f"{row[name]:>{w}{spec}}" for name, w, spec in _COLS))


print_table(RUNS)

## 3. Figure 1 — speed ladder per field point

One row of three panels per point in `POINTS`, log–log, x ticks = the actual $N$ present:
**(a)** in-vivo head time per step vs $N$, one curve per head arm (baseline excluded — it has no
head); **(b)** the head's share of the step, `share = t_head / step_wall` (the published
statistic — plotted as-is, not the `share_noqgt` table variant), same arms; **(c)** total step
time, `baseline` (thick grey) vs every head arm on top of it. One legend for the whole figure.
Panel (b)'s denominator includes the SR solve's iterative CG (`--qgt onthefly`), which is itself
data-dependent per arm/step — a few-percent contribution that is not part of the head; see the
table's `qgt_mean`/`share_noqgt` columns to isolate it.

Two annotations flag points that are not directly comparable to the rest of their own curve: an
**open circle** overlay + a `NN% fallback` text label when `fallback_frac > 2%` (`vote`/`pt2`
partially fall back to the `linear` fixed representative at larger $N$, so part of that point's
cost is really `linear`'s profile); a **hollow diamond** overlay + a `n_samples=NNN` label when a
run's `n_samples` differs from `DEFAULT_N_SAMPLES` (e.g. a reduced batch to fit memory) — its raw
`t_head`/`step_wall` are not on the same footing as the rest of the curve, though the printed
table's `t_head_per_row_us` still is. `savefig` stays commented out.

In [ ]:
def _window_label(runs_here):
    lens = [r["n_steps"] for r in runs_here]
    if not lens:
        return f"steps {PLATEAU_START}+"
    ends = sorted({PLATEAU_START + n - 1 for n in lens})
    tail = "/".join(str(e) for e in ends)          # e.g. 59/99 when run lengths differ across L
    return f"plateau mean, steps {PLATEAU_START}-{tail} (window = index {PLATEAU_START} to end of run)"


def _style_log_N_axis(ax, Ns):
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("qubits $N$")
    ax.xaxis.set_major_locator(FixedLocator(Ns))
    ax.xaxis.set_minor_locator(NullLocator())
    ax.xaxis.set_major_formatter(ScalarFormatter())
    ax.set_xticklabels([str(n) for n in Ns])


def _annotate_special(ax, x, y, r):
    """Overlay + label a point whose fallback rate or batch size makes it not
    directly comparable to the rest of its curve (see the Figure 1 markdown)."""
    notes = []
    ff = r.get("fallback_frac")
    if ff is not None and np.isfinite(ff) and ff > 0.02:
        ax.plot(x, y, "o", mfc="none", mec=ARM_COLOR[r["arm"]], mew=1.6, ms=11, zorder=5)
        notes.append(f"{ff * 100:.0f}% fallback")
    ns = r.get("n_samples")
    if ns is not None and ns != DEFAULT_N_SAMPLES:
        ax.plot(x, y, "D", mfc="none", mec=ARM_COLOR[r["arm"]], mew=1.6, ms=13, zorder=5)
        notes.append(f"n_samples={ns}")
    if notes:
        ax.annotate(", ".join(notes), (x, y), fontsize=6.5, xytext=(4, 4),
                    textcoords="offset points", color=ARM_COLOR[r["arm"]])


def plot_speed_ladder(runs, hx, hz, tol=1e-9):
    here = [r for r in runs if abs(r["hx"] - hx) < tol and abs(r["hz"] - hz) < tol]
    if not here:
        print(f"[fig1] no runs at (hx,hz)=({hx},{hz})")
        return None
    Ns = sorted({r["N"] for r in here})

    fig, (ax_head, ax_share, ax_tot) = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
    for arm in ARMS:
        s = sorted([r for r in here if r["arm"] == arm], key=lambda r: r["N"])
        if not s:
            continue
        N = [r["N"] for r in s]
        base = arm == "baseline"

        y_tot = [r["step_wall"] for r in s]
        ax_tot.plot(N, y_tot, ARM_MARK[arm] + "-", color=ARM_COLOR[arm],
                    lw=2.6 if base else 1.6, ms=9 if base else 6, mew=2.2 if base else 1.2,
                    label=ARM_LABEL[arm])
        for r, y in zip(s, y_tot):
            _annotate_special(ax_tot, r["N"], y, r)

        if not base:
            y_head = [r["t_head"] for r in s]
            y_share = [r["share"] for r in s]
            ax_head.plot(N, y_head, ARM_MARK[arm] + "-", color=ARM_COLOR[arm], lw=1.6, ms=6)
            ax_share.plot(N, y_share, ARM_MARK[arm] + "-", color=ARM_COLOR[arm], lw=1.6, ms=6)
            for r, y in zip(s, y_head):
                _annotate_special(ax_head, r["N"], y, r)
            for r, y in zip(s, y_share):
                _annotate_special(ax_share, r["N"], y, r)

    for ax in (ax_head, ax_share, ax_tot):
        _style_log_N_axis(ax, Ns)
    ax_head.set_ylabel(r"head time per step, $t_{head}$ [s]")
    ax_head.set_title("(a) in-vivo head time per VMC step")
    ax_share.set_ylabel(r"$t_{head}$ / step_wall")
    ax_share.set_title("(b) head share of the step")
    ax_tot.set_ylabel("total VMC step wall-clock [s]")
    ax_tot.set_title("(c) total step: baseline vs GPU + head arms")

    handles, labels = ax_tot.get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=len(labels), frameon=False,
               bbox_to_anchor=(0.5, -0.06), fontsize=9)
    win = _window_label(here)
    fig.suptitle(rf"Speed ladder at $(h_x,h_z)=({hx:g},{hz:g})$ -- {win}", y=1.05)

    out = FIGS / f"fermionic_speed_ladder_hx{hx:g}_hz{hz:g}.png"
    # fig.savefig(out, dpi=300, bbox_inches="tight")
    return fig


for _hx, _hz in POINTS:
    _fig = plot_speed_ladder(RUNS, _hx, _hz)
    if _fig is not None:
        plt.show()

## 4. Figure 2 — stress vs contrast

Same `share` and `k_mean` (mean lit lines per row) as above, but arranged to compare the two
`POINTS` directly: for each head arm, solid = the stress point (`POINTS[0]`), dashed/open = the
contrast point (`POINTS[1]`), restricted to the sizes $N$ where **both** points have a surviving
run for that arm (so every pair of curves shares its domain).

In [ ]:
def plot_stress_vs_contrast(runs, points=POINTS, tol=1e-9):
    if len(points) < 2:
        print("[fig2] need at least 2 POINTS")
        return None
    p0, p1 = points[0], points[1]

    def by_arm_N(pt):
        hx, hz = pt
        out = {}
        for r in runs:
            if abs(r["hx"] - hx) < tol and abs(r["hz"] - hz) < tol:
                out.setdefault(r["arm"], {})[r["N"]] = r
        return out

    g0, g1 = by_arm_N(p0), by_arm_N(p1)
    head_arms = [a for a in ARMS if a != "baseline"]

    fig, (ax_share, ax_k) = plt.subplots(1, 2, figsize=(12.5, 5), constrained_layout=True)
    any_plotted, all_Ns = False, set()
    for arm in head_arms:
        d0, d1 = g0.get(arm, {}), g1.get(arm, {})
        common_N = sorted(set(d0) & set(d1))
        if not common_N:
            continue
        any_plotted = True
        all_Ns.update(common_N)
        ax_share.plot(common_N, [d0[n]["share"] for n in common_N], ARM_MARK[arm] + "-",
                      color=ARM_COLOR[arm], lw=1.6, ms=6)
        ax_share.plot(common_N, [d1[n]["share"] for n in common_N], ARM_MARK[arm] + "--",
                      color=ARM_COLOR[arm], lw=1.6, ms=6, mfc="none")
        ax_k.plot(common_N, [d0[n]["k_mean"] for n in common_N], ARM_MARK[arm] + "-",
                  color=ARM_COLOR[arm], lw=1.6, ms=6)
        ax_k.plot(common_N, [d1[n]["k_mean"] for n in common_N], ARM_MARK[arm] + "--",
                  color=ARM_COLOR[arm], lw=1.6, ms=6, mfc="none")

    if not any_plotted:
        print("[fig2] no size overlaps between the two POINTS yet")
        plt.close(fig)
        return None

    Ns = sorted(all_Ns)
    for ax in (ax_share, ax_k):
        _style_log_N_axis(ax, Ns)
    ax_share.set_ylabel(r"$t_{head}$ / step_wall")
    ax_share.set_title("(a) head share: stress vs contrast")
    ax_k.set_ylabel("mean lit lines per row")
    ax_k.set_title("(b) lit-line load: stress vs contrast")

    arm_handles = [Line2D([0], [0], color=ARM_COLOR[a], marker=ARM_MARK[a], lw=1.6, label=ARM_LABEL[a])
                   for a in head_arms]
    style_handles = [Line2D([0], [0], color="0.2", ls="-", label=f"stress {p0}"),
                     Line2D([0], [0], color="0.2", ls="--", label=f"contrast {p1}")]
    fig.legend(handles=arm_handles + style_handles, loc="lower center",
               ncol=len(arm_handles) + 2, frameon=False, bbox_to_anchor=(0.5, -0.1), fontsize=8)
    fig.suptitle("Figure 2 -- stress vs contrast: head share and lit-line load vs $N$", y=1.05)

    out = FIGS / "fermionic_speed_ladder_stress_vs_contrast.png"
    # fig.savefig(out, dpi=300, bbox_inches="tight")
    return fig


_fig2 = plot_stress_vs_contrast(RUNS)
if _fig2 is not None:
    plt.show()

## 5. What these panels mean

- **(a) head time per step** and **(b) head share** both come from the same per-step pair
  (`t_head`, `total`), so shared-node noise on the timing ladder mostly cancels: a node that runs
  20-40% slower (or faster) than another moves both numbers together, and their *ratio* is close
  to invariant. **(c) total step time** is a raw wall-clock number, not a ratio — it carries that
  node-to-node noise directly, so read absolute totals as "same order of magnitude", and read
  arm-vs-arm *shape* rather than precise multiplicative factors, unless the compared points ran
  on the same job/node.
- **`t_head_per_row_us`** (printed table only) divides out `n_samples` entirely, so it is the one
  column that stays comparable across a batch-size change (e.g. a `pt2` point run at a reduced
  `n_samples` to fit memory) where the raw `t_head`/`step_wall`/panel curves are not — see the
  hollow-diamond annotation on Figure 1.
- **`qgt_mean`/`share_noqgt`**: the SR solve is an iterative CG (`--qgt onthefly`), so the `qgt`
  phase itself is data-dependent per arm/step, not a fixed overhead. `share` (plotted in panel
  (b)) is the published statistic and includes it; `share_noqgt = mean(t_head) / mean(total -
  qgt)` divides it back out, for when the CG noise itself (typically a few percent) needs to be
  ruled out as the explanation for a share difference between arms.
- **`step_wall_median`/`t_head_median`/`outliers`**: reported alongside the plateau means because
  a mean is not robust to a one-off stall. `outliers` counts window entries whose `total` exceeds
  5x the window's median `total` — a mid-run JIT recompile is exactly this signature (one entry
  far above an otherwise flat plateau); a nonzero `outliers` with `step_wall_median` well below
  `step_wall` (the mean) is the tell.
- **Acceptance for a run to enter the table**: a final json wins over its `.curve.json` twin
  whenever one exists (the `.curve.json` always lags the true timing list by one entry, since
  `tc3d/train.py` writes it inside `on_step`, before that step's `on_timing` call); a run falls
  back to its `.curve.json` only when no final json exists at all, which is the `walltime_cut`
  signature (kept and flagged, not dropped). Independently, at least 5 timing entries at/after
  `PLATEAU_START` (step index 20) are required — fewer than that is the JIT/transient tail, not a
  plateau. `diverged` (from the final json; `?` when unknown for a `walltime_cut` run) is a
  separate flag: a run can finish and still be marked `diverged` if the training-time guard
  stopped it early.
- **`drift_pct`** flags a head whose cost is still climbing inside the window (e.g. a tie-break
  search whose load grows with a heavy-tailed defect count) — a plateau mean is only a fair
  single number when `drift_pct` is small; a large drift means the mean understates where the arm
  is heading and the trend should be reported alongside it.
- **`fallback_frac`**: `vote`/`pt2` are decoded heads that fall back to the `linear` fixed
  representative when their own recovery search comes up empty; a fallback fraction above a
  couple of percent (annotated on Figure 1) means a meaningful slice of that arm's samples are
  actually running `linear`'s cost profile, not its own.